# Table of Contents

1. [Bronze Layer](#bronze-layer)
2. [First ETL Steps](#first-etl-steps)
   - [Data extraction using Volumes](#data-extraction-using-volumes)
   - [Import using SQL](#import-using-sql)
   - [Import using PySpark](#import-using-pyspark)
   - [Data extraction using tables](#data-extraction-using-tables)
3. [Quick Schema & Sample Data Checks](#quick-schema--sample-data-checks)
4. [Bronze Layer Quick Notes](#bronze-layer-quick-notes)
5. [External Tables](#external-tables)
6. [Detecting Duplicate Keys in the Bronze Layer](#detecting-duplicate-keys-in-the-bronze-layer)
7. [Bronze Layer Notes - Order Reviews](#bronze-layer-notes---order-reviews)
8. [Missing Value Audit](#missing-value-audit)
9. [Olist - Bronze Missing Value Profile](#olist---bronze-missing-value-profile)
10. [Final Data Checks before Moving to Silver Layer](#final-data-checks-before-moving-to-silver-layer)
11. [Using SQL Queries for Data Profiling](#using-sql-queries-for-data-profiling)
12. [Cleaning and Normalizing Customers Table](#cleaning-and-normalizing-customers-table)

# BRONZE LAYER
This is where the rew data lives. Data extracted from the external source lives here. We will have data from brazilian ecommerce firm olist which has data of about 100K orders placed between 2016 and 2018. This is real data, but data privacy has been taken care to anonamize the customer and store names. Also, company names have been replaced by the house names of the major TV show 'Game of Thrones'.

Data for the [ecommerce company](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) is from Kaggle. There were a total of 9 csv files that were downloaded and tables were created in Datarbricks for each of these files. So, the catalog is course_training_catalog and the scema is bronze and then the table name follows.

Tables allows us to run SQL queries, perform cleaning, make transformations on the data, this is good for strucutred data. 
But, for semi structured or unstructured data, we can use volumes which is a type of file storage system provided by databricks, spark will then be used to perform operations. In these examples, we will use both tables and volumes to read the data. 


## First ETL Steps 
## Data extraction using Volumes
### 1. Import using sql

In [0]:
%sql
SELECT * FROM read_files("/Volumes/course_training_catalog/bronze_olist/olist_volumes/orders/orders.csv",format => "csv");

### 2. Import using PySpark

In [0]:
df_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("dbfs:/Volumes/course_training_catalog/bronze_olist/olist_volumes/orders/orders.csv")

In [0]:
display(df_orders)

In [0]:
base_path = "/Volumes/course_training_catalog/bronze_olist/olist_volumes"

#Customers
df_customers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{base_path}/customers/customers.csv")

#Products
df_products = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/products.csv")

#Order Items
df_order_items = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_items.csv")

#Order Payments
df_order_payments = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_payments.csv")

#Order Review Ratings
df_order_reviews = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_reviews.csv")

# #Orders
# df_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load
# #(base_path+"/orders/orders.csv")

#Sellers
df_sellers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/sellers/sellers.csv")

#Geolocation
df_geolocation = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/customers/geolocation.csv")

# Product Category Translation
df_categories = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/product_category_name_translation.csv") 



In [0]:
# To print the schema definition of a data frame
df_orders.printSchema()
# df_products.printSchema()
# df_order_items.printSchema()
# df_order_payments.printSchema()
# df_order_reviews.printSchema()
# df_sellers.printSchema()
# df_geolocation.printSchema()
# df_categories.printSchema()
# df_customers.printSchema()


## Data extraction using tables


In [0]:
df_table_products = spark.table("course_training_catalog.bronze_olist.products")

In [0]:
df_table_products.printSchema()

In [0]:
df_table_products.summary()

In [0]:
df_table_products.count()

In [0]:
# Extract rest of the tables apart from Products
df_table_customers = spark.table("course_training_catalog.bronze_olist.customers")
df_table_geolocation = spark.table("course_training_catalog.bronze_olist.geolocation")
df_table_sellers = spark.table("course_training_catalog.bronze_olist.sellers")
df_table_order_items = spark.table("course_training_catalog.bronze_olist.order_items")
df_table_order_payments = spark.table("course_training_catalog.bronze_olist.order_payments")
df_table_order_reviews = spark.table("course_training_catalog.bronze_olist.order_reviews")
df_table_orders = spark.table("course_training_catalog.bronze_olist.orders")
df_table_categories = spark.table("course_training_catalog.bronze_olist.product_category_name_translation")
# df_table_customers.count()
# df_table_geolocation.count()
# df_table_sellers.count()
# df_table_order_items.count()
# df_table_order_payments.count()
# df_table_order_reviews.count()
# df_table_orders.count()
# df_table_categories.count()
# df_table_products = spark.table("course_training_catalog.bronze_olist.products")


In [0]:
df_table_orders.count()

## Quick Schema & Sample Data Checks

In [0]:
# Put all dataframes in a dictionary
dataframes = {
    "orders": df_orders,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "geolocation": df_geolocation,
    "categories": df_categories,
    "customers": df_customers,
}

for name, df in dataframes.items():
    print(f"\n======={name.upper()}=======")
    df.printSchema()
    df.summary().show()
    print(f"Number of records: {df.count()}")

## Bronze Layer Quick Notes

### **1. Orders**
* 8 columns; 'order_id' and 'customer_id' are strings; all date columns correctly inferred as *timestamp*
* **key:** 'order_id' is the primary key - check for duplicates
* **Critical column:** 'order_status' is categorical (delivered,shipped,cancelled,...). Distribution should be inspected
* Some timestamp columns ('order_approved_at', 'order_delivered_customer_date') may contain null values, handle in silver

### **2. Customers**
* 5 columns; 'customer_id' is string, 'customer_zip_code_prefix' is integer
* **key:** customer_id should be unique. but the same 'customer_unique_id' can map to mulitple 'customer_id's
* **Critical Columns:** 'customer_city', 'customer_state' - distributions worth checking
* Zip codes stored as integers - useful for address analysis

### **3. Order items**
* contains line-item details; links to orders, products and sellers
* **key:** combination of 'order_id' + 'order_item_id' should be unique
* 'price' and 'freight_value' are doubles - check for nulls/zeroes
* 'shipping_limit_date' correctly inferred as timestamp

### **4. Product Category Translation**
* 2 columns - Portuguese and English category names
* Acts as a lookup table to standardize category names
* Just check the structure in Bronze, join with producst in Silver

### **5. Products**
* 9 columns; product dimensions, weight, description length, phot qty all inferred as integers
* **Key:** 'product_id' should be unique, check for duplicates
* **Critical column** 'product_category_name'. number of categories and distribution should be noted
* weight/size fields may contain zero or extremem values - clean later in Silver

### **6. Sellers**
* 4 columns; 'seller_id' string, 'seller_zip_code_prefix' integer
* **key:** 'seller_id' should be unique
* city/state fields cn abe used for distribution checks
* zip code info can be cross-checked with customers for address consistency in silver. 


### **7. Gelocations**
* contains geocordinates linked to zip prefixes
* no unique key: multiple coordinates may exist per prefix
* 'geolocation_lat', 'geolocation_lng' are doubles - check for null values
* city/state fields should be cross-checked with customers/sellers in silver.
* functions as a lookup table for address/location analysis

### **8. Order Payments**
* Payment details; linked to orders via order_id
* **key: multiple rows per order possible --> 'payment_sequential' used as ordering field.
* **Critical Column** 'payment_type' (credit card, boleto, voucher,...) distribution analysis recommended
* 'payment_value' is double - check for consistency with total order value in silver
* 'payment_installments' important for installment analysis.

### **9. Order Reviews**
* customer reviews; linked to orders via 'order_id'
* **key:** 'review_id' should be unique
* **Critical column** 'review_score' - important for quality analysis but inferred as string; convert to integer in silver
* 'review_creation_date' and 'review_answer_timestamp' are strings --> convert to timestamp in silver



# External Tables 
##Reading a table from an external storage

CREATE TABLE IF NOT EXISTS catalog.schema.logs (
	log_id			INT,
	user_id			STRING,
	log_date			DATE,
	log_message		STRING
)
USING CSV
OPTIONS (
	header “false”,	— no header in the file
	delimiter “,”		— delimiter (comma is default, so this is optional)
)
LOCATION ‘abfss://external-data@companystorage.dfs.core.windows.net/logs/';


###To check the metadata of the table
DESCRIBE EXTENDED catalog.schema.logs;

### if the data is not refreshed, use the following command to do a refresh
REFRESH TABLE catalog.schema.logs;

### To drop a table
DROP TABLE catalog.schema.logs;   — only deletes the metadata in unity catalog

## Detecting duplicate keys in the Bronze layer

In [0]:
###########################################################################################################################################
base_path = "/Volumes/course_training_catalog/bronze_olist/olist_volumes"
#Orders
df_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/orders.csv")
#Customers
df_customers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{base_path}/customers/customers.csv")
#Sellers
df_sellers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/sellers/sellers.csv")
#Products
df_products = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/products.csv")
# Product Category Translation
df_categories = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/product_category_name_translation.csv") 
#Order Items
df_order_items = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_items.csv")
#Order Payments
df_order_payments = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_payments.csv")
#Order Review Ratings
df_order_reviews = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_reviews.csv")
#Geolocation
#df_geolocation = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/customers/geolocation.csv")

##########################################################################################################################
dataframes = {
    "orders": (df_orders, ["order_id"]),
    "customers": (df_customers, ["customer_id"]),
    "sellers": (df_sellers, ["seller_id"]),
    "products": (df_products, ["product_id"]),
    "product_category": (df_categories, ["product_category_name"]),
    "order_items": (df_order_items, ["order_id", "order_item_id"]),
    "order_payments": (df_order_payments, ["order_id", "payment_sequential"]),
    "order_reviews": (df_order_reviews, ["review_id"])
}

# Simple duplicate check
for name, (df, keys) in dataframes.items():
    total = df.count()
    distinct = df.select(*keys).distinct().count()
    print(f"{name.upper()}: total = {total}, distinct = {distinct}, duplicates = {total-distinct}")


## Bronze Layer Notes - Order Reviews
- 1204 duplicates detected in hte review_id column. 
- This means that the same review_id appears in more than one row
### Possible reasons:
- Records might have ben written multiple times during data loading. 
- or diferent versions of the same review may have been stored. 
In the Bronze layer, we only record this issue, no cleaning is performed yet. 

In the Silver layer, the duplicates must be addressed by
- Removing exact duplicate records,
- considering the combination of review_id + order_id as the key.
- or keeping only the most recent/consistent version of the review. 

## Missing Value Audit

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import(DoubleType, FloatType, LongType, ShortType, DecimalType)
###########################################################################################################################################
base_path = "/Volumes/course_training_catalog/bronze_olist/olist_volumes"
#Customers
df_customers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{base_path}/customers/customers.csv")
#Products
df_products = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/products.csv")
#Order Items
df_order_items = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_items.csv")
#Order Payments
df_order_payments = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_payments.csv")
#Order Review Ratings
df_order_reviews = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_reviews.csv")
#Orders
df_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/orders.csv")
#Sellers
df_sellers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/sellers/sellers.csv")
#Geolocation
df_geolocation = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/customers/geolocation.csv")
# Product Category Translation
df_categories = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/product_category_name_translation.csv") 
###########################################################################################################################################
# Put all dataframes in a dictionary
dataframes = {
    "orders": df_orders,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "geolocation": df_geolocation,
    "categories": df_categories,
    "customers": df_customers,
}

In [0]:
def safe_missing_report(df):
    """
    Missing = NULL + trimmed empty string ("") + NaN (only for numeric columns)
    Returns a Spark DF with columns: ["column", "missing_count", "missing_ratio"]
    """

    total = df.count()
    rows = []

    for field in df.schema.fields:
        c = field.name
        dt = field.dataType

        # NULL and empty string checks for all columns
        is_null = F.col(c).isNull()
        is_empty = (F.trim(F.col(c).cast("string")) == "")

        # NaN check only for float/Double types
        if isinstance(dt, (DoubleType, FloatType)):
            is_nan = F.isnan(F.col(c))
        
        else:
            # No NaN concept for Integer/Long?Short/Decimal/String
            is_nan = F.lit(False)

        cond = is_null | is_empty | is_nan
        missing = df.filter(cond).count()
        ratio = (missing / total) if total > 0 else 0.0
        rows.append((c, int(missing), float(ratio)))

    return spark.createDataFrame(rows, ["column", "missing_count", "missing_ratio"])
    
# Run for all tables
for name, df in dataframes.items():
    print(f"\n===={name.upper()}====")
    safe_missing_report(df).orderBy("missing_count", ascending = False).show(truncate = False)

## Olist - Bronze Missing Value Profile
### Goal & Method
- **Goal:** In the Bronze layer, profile missing values without modifying data to plan precise cleanups/standardization in Silver. 
- "Missing" definitino:
  -   NULL
  -   Trimmed empty string ("" or whitespace-only)
  -   NaN (only for Float/Double columns)

### Quick Summary
- Orders: Expected gaps on delivery timestamps (to-customer ~ 2.98%, to-carruer ~ 1.79%)
- Customers/Sellers/Geolocation/Order Items/Order Payments/Category Translation: no missing values (or neglible)
- Products: Category/summary fields ~ 1.85% missing; physical dimensions missing in ~0.006% of rows(2 products)
- Order Reviews: Text fields very sparse (title~88.48%, message~60.57% missing), Also order_id ~ 2.15% and review_score ~ 2.28% missing. 

------------------------------------------------------------------------------------------------------------------------
### Table-by-Table details
**1) Orders**
For order_delivered_customer_date there are 2966 missing ~2.98%
For order_dlivered_carrier_date there are 1783 missing ~1.79%
For order_approved_at there are 160 missing ~0.16%

interpetation: delivery gaps typically come from cancelled/not-delivered orders. 
Silver note: compute delivery/latency metrics only for relevant statuses(e.g, delivered) and non-null timestamps. Cast time fields to prpoer timestamp. 

**2) Customers, sellers, geolocation, order_items, order_payments, product_cateogry_name_translation**
- Missing: none/negligible
- Interpretation: strong join backbone; dimensional references look clean
- Silver note: geolocation can be multi-row per prefix, if you need a single city/state per prefix, standardize with a clear rule (e.g., most-frequent

**3) Products**
- for the columns product_category_name, *_name_length, *_description_length, product_photos_qty 610 missing ~1.85%
- prodcut_weight_g, prodcut_length_cm, prodcut_height_cm, product_width_cm 2 missing ~ 0.006%

Inerpretation: Category/Summary fields have minor gaps, physical measurement are almost fully populated. 

Silver Note: 
- Fix naming consistency: *_lenght --> *.length (presever lineage to orginal Bronze names)
- Cast numeric fields appropriately; Keep NULLs for measurements; optional category-level median imputation can be tested separately. 

**4)Order_reviews**
-  For review_comment_title 92159 missing ~88.48%
- review_comment_message 63088 missing ~60.57%
- review_answer_timestamp 8785 missing ~8.43%
- review_creation_date 8764 missing ~8.41%
- review_score 2380 missing ~2.28%
- order_id 2240 missing ~2.15%
- review_id 1 missing ~0.001%


------------------------------------------------------------------------------------------------------------------
### Siilver Layer Action Plan
1. Types & Naming
- Cast timestamps/numerics to correct types across all tables. 
- Standardize product name fields (*_length),while maintanining lineage to Bronze names. 

2. Missing-Data strategy
- Orders; calcualted delivery KPI's only on relevant statuses and complete timestamps
- Reviews: Exclude rows with NULL Order_id/review_score from resepctive analyses, limit NLP to non-empty text. 
- Products: Leave measurement NULLs, optionally test median-by-category imputation (report impact).

3. Data Quality (DQ) Checks/Expectations
- order.order_id unique; order_itmes.order_id all exist in orders
- order_reviews.review_id unique; track % NULL order_id and % NULL review score.
- Persist the missing-value profile as a Delta table for monitoring over time. 

## Final Data checks before moving to Silver layer


In [0]:
display(df_orders)
# dbutils.data.summarize(df_orders)
# note that dbutils is not availabe in serverless compute. This is a Databricks specific function

Databricks visualization. Run in Databricks to view.

## Using SQL queries to do some data profiling

In [0]:
%sql
-- Count total rows and missing columns in customers
SELECT 
  COUNT(*) AS total_rows,
  COUNT(customer_id) AS filled_customer_id,
  COUNT_IF(customer_id IS NULL) AS missing_customer_id
FROM course_training_catalog.bronze_olist.customers;

In [0]:
%sql
-- Check duplicates in orders
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS unique_order_ids,
  COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_orders
FROM course_training_catalog.bronze_olist.orders;


In [0]:
%sql
-- Basic statistics for delivery dates
SELECT 
  MIN(order_purchase_timestamp) AS min_purchase_date,
  MAX(order_purchase_timestamp) AS max_purchase_date,
  COUNT_IF(order_purchase_timestamp IS NULL) AS missing_purchase_date
FROM course_training_catalog.bronze_olist.orders

In [0]:
%sql
-- Frequency of order statuses
SELECT
  order_status,
  COUNT(*) AS count
FROM course_training_catalog.bronze_olist.orders
GROUP BY order_status
ORDER BY count DESC;

## Cleaning and Normalizing Customers table

In [0]:
%sql
USE CATALOG course_training_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver_olist;

In [0]:
display(df_customers)

In [0]:
df_customers.describe().show()

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_base AS
SELECT DISTINCT
  customer_id,
  customer_unique_id,
  customer_zip_code_prefix,
  customer_city,
  customer_state
FROM course_training_catalog.bronze_olist.customers
WHERE customer_id IS NOT NULL;

In [0]:
%sql
SELECT * FROM customer_base LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_norm AS
SELECT 
  customer_id,
  customer_unique_id,
  CAST(customer_zip_code_prefix AS INT) AS customer_zip_code_prefix,
  LOWER(TRIM(regexp_replace(customer_city, '\s+', ' '))) AS customer_city,
  UPPER(TRIM(regexp_replace(customer_state, '\s+', ' '))) AS customer_state
FROM customer_base;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_olist.customers
USING DELTA
AS
SELECT
  customer_id,
  customer_unique_id,
  customer_zip_code_prefix,
  customer_city,
  customer_state
FROM customers_norm;

In [0]:
%sql
COMMENT ON TABLE course_training_catalog.silver_olist.customers IS
'Source: bronze_olist.customers
Silver transformations: PK hygiene (DISTINCT, non-null), city lowercase+trim, state uppercase+trim, Zip INT cast
Purpose: clean, analysis-ready customer dimension'

In [0]:
%sql
DESCRIBE EXTENDED course_training_catalog.silver_olist.customers;

In [0]:
%sql
SELECT COUNT(*) FROM course_training_catalog.silver_olist.customers;